# Polymarket Sports Market Data for Arbitrage (Demo)

### Install dependencies

Installs packages needed

- `websocket-client` for the public WebSocket stream
- `pandas`, `requests` for REST calls and tables



In [ ]:
!pip -q install websocket-client pandas requests


### Imports + endpoints

- REST bases:
  - `GAMMA` (market metadata: sports, events, markets)
  - `CLOB`  (order book + price endpoints)
- `WSS_MARKET` public WebSocket channel



In [ ]:
import json
import time
import math
import requests
import pandas as pd
from datetime import datetime, timezone

GAMMA = "https://gamma-api.polymarket.com"
CLOB  = "https://clob.polymarket.com"
WSS_MARKET = "wss://ws-subscriptions-clob.polymarket.com/ws/market"  # public market channel

session = requests.Session()
session.headers.update({"User-Agent": "colab-polymarket-tutorial/1.0"})


### Helper utilities

Defines helpers:

- `get_json(...)`: GET with retries
- `maybe_json_list(...)`: Gamma sometimes returns arrays as JSON-encoded strings; this makes them consistent
- `ts_utc()`: convenience timestamp

In [ ]:
def get_json(url: str, params: dict | None = None, timeout: int = 20, retries: int = 3):
    last_err = None
    for attempt in range(retries):
        try:
            r = session.get(url, params=params, timeout=timeout)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            last_err = e
            time.sleep(1.5 * (attempt + 1))
    raise RuntimeError(f"GET failed after {retries} tries: {url} params={params}\n{last_err}")

def maybe_json_list(x):
    """Gamma sometimes returns arrays as JSON-encoded strings; handle both."""
    if x is None:
        return None
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        try:
            v = json.loads(x)
            return v if isinstance(v, list) else x
        except Exception:
            return x
    return x

def ts_utc():
    return int(datetime.now(timezone.utc).timestamp())


### List available sports / leagues (Gamma)

Calls `GET /sports` from Gamma and shows the first rows as a DataFrame.

This is where you can pick  `series_id` .

In [ ]:
sports = get_json(f"{GAMMA}/sports")  # official endpoint
df_sports = pd.DataFrame(sports)
df_sports.head(20)


,id,sport,image,resolution,ordering,tags,series,createdAt
0,1,ncaab,https://polymarket-upload.s3.us-east-2.amazona...,https://www.ncaa.com/march-madness-live/bracket,home,"1,100149,100639",39,2025-11-05T19:27:45.399303Z
1,2,epl,https://polymarket-upload.s3.us-east-2.amazona...,https://www.premierleague.com/,home,"1,82,306,100639,100350",10188,2025-11-05T19:27:45.399303Z
2,3,lal,https://polymarket-upload.s3.us-east-2.amazona...,https://www.laliga.com/en-GB,home,"1,780,100639,100350",10193,2025-11-05T19:27:45.399303Z
3,95,acn,https://polymarket-upload.s3.us-east-2.amazona...,https://www.cafonline.com/caf-africa-cup-of-na...,home,"1,102974",10786,2025-12-18T19:19:31.18134Z
4,5,ipl,https://polymarket-upload.s3.us-east-2.amazona...,https://www.iplt20.com/,home,"1,101977,100639,517,518",44,2025-11-05T19:27:45.399303Z
5,6,wnba,https://polymarket-upload.s3.us-east-2.amazona...,https://www.wnba.com/,away,"1,100639,100254",10105,2025-11-05T19:27:45.399303Z
6,7,bun,https://polymarket-upload.s3.us-east-2.amazona...,https://www.bundesliga.com/en/bundesliga,home,"1,1494,100639,100350",10194,2025-11-05T19:27:45.399303Z
7,8,mlb,https://polymarket-upload.s3.us-east-2.amazona...,https://www.mlb.com/,away,"1,100639,100381",3,2025-11-05T19:27:45.399303Z
8,9,cfb,https://polymarket-upload.s3.us-east-2.amazona...,https://www.ncaa.com/,away,"1,100351,100639",10210,2025-11-05T19:27:45.399303Z
9,10,nfl,https://polymarket-upload.s3.us-east-2.amazona...,https://www.nfl.com/,away,"1,450,100639",10187,2025-11-05T19:27:45.399303Z


### Fetch events for a chosen series

Calls `GET /events` with parameters like:

- `series_id`
- `active=true`
- `closed=false`
- `limit=...`

Then builds `df_events` (event title, start time, number of markets).

In [ ]:
# Pick a league/series from df_sports (edit this for different series)
SERIES_ID = 10187

params = {
    "series_id": SERIES_ID,
    "active": "true",
    "closed": "false",
    "limit": 10
}
events = get_json(f"{GAMMA}/events", params=params)

# NOTICE if empty
if not events:
    print(f"⚠️ No active/open events found for series_id={SERIES_ID}.")
    print("Try a different series_id from df_sports, or remove filters (active/closed) to broaden results.")
    df_events = pd.DataFrame(columns=["event_id", "title", "startTime", "num_markets"])
else:
    df_events = pd.DataFrame([{
        "event_id": e.get("id"),
        "title": e.get("title"),
        "startTime": e.get("startTime"),
        "num_markets": len(e.get("markets", []) or []),
    } for e in events])

df_events


,event_id,title,startTime,num_markets
0,174405,Patriots vs. Broncos,2026-01-25T20:00:00Z,73
1,174407,Rams vs. Seahawks,2026-01-25T23:30:00Z,59


###  Extract outcomes + CLOB token IDs for specific market

Chooses an event that has at least one market, then grabs the first market.

Prints:
- event title
- market question
- outcomes + current outcomePrices (from Gamma)
- `clobTokenIds` (the CLOB token IDs you use for pricing / order books)

In [ ]:
# Pick an event that has at least 1 market
event = next(e for e in events if (e.get("markets") or []))
market = (event.get("markets") or [])[0]

outcomes = maybe_json_list(market.get("outcomes"))
outcome_prices = maybe_json_list(market.get("outcomePrices"))

# IMPORTANT: clobTokenIds can be a JSON-encoded string like '["...","..."]'
token_ids = maybe_json_list(market.get("clobTokenIds"))

print("Event:", event.get("title"))
print("Market question:", market.get("question"))
print("Outcomes:", outcomes)
print("OutcomePrices:", outcome_prices)
print("clobTokenIds:", token_ids, "| type:", type(token_ids))

if not isinstance(token_ids, list) or len(token_ids) == 0:
    raise ValueError(f"⚠️ clobTokenIds is missing or not a list. Got: {token_ids}")

# Convenience: choose the first token id (often "Yes" if outcomes=[Yes,No])
TOKEN_ID = str(token_ids[0])
print("Using TOKEN_ID:", TOKEN_ID)


Event: Patriots vs. Broncos
Market question: Patriots vs. Broncos
Outcomes: ['Patriots', 'Broncos']
OutcomePrices: ['0.435', '0.565']
clobTokenIds: ['93940441597806605185863489141137222482434084554690688125580295758685733000625', '84067471748088275158649178487081940954142612413002471523992575056649818081482'] | type: <class 'list'>
Using TOKEN_ID: 93940441597806605185863489141137222482434084554690688125580295758685733000625


###  Snapshot: current price + order book (REST)

Uses CLOB REST endpoints to get real time prices

This is a **point-in-time snapshot**. If you run it again, you’ll get an updated snapshot.

In [ ]:
# Current prices (note: side is BUY / SELL)
buy_px  = get_json(f"{CLOB}/price", params={"token_id": TOKEN_ID, "side": "BUY"})
sell_px = get_json(f"{CLOB}/price", params={"token_id": TOKEN_ID, "side": "SELL"})

book = get_json(f"{CLOB}/book", params={"token_id": TOKEN_ID})

print("BUY price:", buy_px)
print("SELL price:", sell_px)
print("Top of book bids:", (book.get("bids") or [])[:5])
print("Top of book asks:", (book.get("asks") or [])[:5])


BUY price: {'price': '0.44'}
SELL price: {'price': '0.45'}
Top of book bids: [{'price': '0.01', 'size': '67979'}, {'price': '0.02', 'size': '55162'}, {'price': '0.03', 'size': '31882'}, {'price': '0.04', 'size': '21368'}, {'price': '0.05', 'size': '23648.22'}]
Top of book asks: [{'price': '0.99', 'size': '85335.87'}, {'price': '0.98', 'size': '44343'}, {'price': '0.97', 'size': '38059'}, {'price': '0.96', 'size': '16901'}, {'price': '0.95', 'size': '950'}]


### Live updates: public WebSocket stream

Starts a WebSocket subscription for one or more token IDs and prints a clean “live feed”.

Why WS if you already have `/price`?
- WS is **push-based** (no polling loop / less API load)
- You get near-real-time updates without repeatedly calling REST endpoints


In [ ]:
# Polymarket WS — print BOTH outcome tokens on one line (clean)

!pip -q install websocket-client
import websocket, threading, json, time
from datetime import datetime, timezone

TOKENS = [str(t) for t in token_ids]   # token_ids from your Gamma event/market cell (length 2)
OUTCOME_NAMES = outcomes               # outcomes from your Gamma cell (e.g., ["Patriots","Broncos"])

state = {t: {"bid": None, "ask": None, "last": None} for t in TOKENS}
last_print_t = 0.0

def fmt(x): return "None" if x is None else f"{float(x):.4f}"

def as_levels(x):
    out = []
    if not x: return out
    for lvl in x:
        if isinstance(lvl, (list, tuple)) and len(lvl) >= 2:
            try: out.append((float(lvl[0]), float(lvl[1])))
            except: pass
        elif isinstance(lvl, dict):
            try: out.append((float(lvl.get("price")), float(lvl.get("size"))))
            except: pass
    return out

def normalize_events(message):
    try:
        obj = json.loads(message)
    except Exception:
        return []
    if isinstance(obj, list):
        return [x for x in obj if isinstance(x, dict)]
    if isinstance(obj, dict):
        d = obj.get("data")
        if isinstance(d, list):
            out = []
            for item in d:
                if isinstance(item, dict):
                    item = dict(item)
                    item.setdefault("event_type", obj.get("event_type") or obj.get("type"))
                    out.append(item)
            return out
        if isinstance(d, dict):
            item = dict(d)
            item.setdefault("event_type", obj.get("event_type") or obj.get("type"))
            return [item]
        return [obj]
    return []

def print_line():
    ts = datetime.now(timezone.utc).strftime("%H:%M:%S")
    parts = []
    for name, tid in zip(OUTCOME_NAMES, TOKENS):
        s = state[tid]
        parts.append(f"{name[:10]:<10} {fmt(s['bid'])}/{fmt(s['ask'])} last {fmt(s['last'])}")
    print(f"[{ts} UTC] " + " | ".join(parts))

def run_ws(print_every=1.0, ping_every=10):
    global last_print_t

    def on_open(ws):
        ws.send(json.dumps({"assets_ids": TOKENS, "type": "market"}))
        def ping_loop():
            while True:
                try:
                    ws.send("PING")
                    time.sleep(ping_every)
                except Exception:
                    break
        threading.Thread(target=ping_loop, daemon=True).start()
        print("✅ Polymarket WS subscribed (both outcomes). Streaming... (stop the cell to end)")

    def on_message(ws, message):
        global last_print_t
        if message == "PONG":
            return

        changed = False
        for ev in normalize_events(message):
            try:
                et = ev.get("event_type") or ev.get("type")
                tid = ev.get("asset_id") or ev.get("token_id") or ev.get("assetId") or ev.get("tokenId")
                tid = str(tid) if tid is not None else None
                if tid not in state:
                    continue

                if et in ("book", "orderbook", "book_change"):
                    bids = as_levels(ev.get("bids") or [])
                    asks = as_levels(ev.get("asks") or [])
                    if bids:
                        bid = max(bids, key=lambda t: t[0])[0]
                        if bid != state[tid]["bid"]:
                            state[tid]["bid"] = bid
                            changed = True
                    if asks:
                        ask = min(asks, key=lambda t: t[0])[0]
                        if ask != state[tid]["ask"]:
                            state[tid]["ask"] = ask
                            changed = True

                elif et in ("last_trade_price", "last_trade", "price_change"):
                    px = ev.get("price") or ev.get("last_trade_price") or ev.get("lastTradePrice") or ev.get("new_price") or ev.get("newPrice")
                    if px is not None:
                        px = float(px)
                        if px != state[tid]["last"]:
                            state[tid]["last"] = px
                            changed = True
            except Exception:
                continue

        now = time.time()
        if changed and (now - last_print_t >= print_every):
            print_line()
            last_print_t = now

    def on_error(ws, err):
        print("WS error:", err)

    def on_close(ws, code, msg):
        print("WS closed:", code, msg)

    ws = websocket.WebSocketApp(
        WSS_MARKET,
        on_open=on_open,
        on_message=on_message,
        on_error=on_error,
        on_close=on_close
    )
    ws.run_forever()

run_ws(print_every=1.0)


✅ Polymarket WS subscribed (both outcomes). Streaming... (stop the cell to end)
[21:24:56 UTC] Patriots   0.5800/0.5900 last None | Broncos    0.4100/0.4200 last None
[21:24:58 UTC] Patriots   0.5800/0.6000 last None | Broncos    0.4100/0.4200 last None
[21:25:05 UTC] Patriots   0.5800/0.6000 last 0.6000 | Broncos    0.4000/0.4200 last 0.4200
[21:25:09 UTC] Patriots   0.5800/0.5900 last 0.6000 | Broncos    0.4000/0.4200 last 0.4200
[21:25:13 UTC] Patriots   0.5700/0.5900 last 0.5800 | Broncos    0.4100/0.4200 last 0.4100
[21:25:17 UTC] Patriots   0.5700/0.5800 last 0.5800 | Broncos    0.4100/0.4300 last 0.4300
[21:25:21 UTC] Patriots   0.5700/0.5800 last 0.5800 | Broncos    0.4200/0.4300 last 0.4200
[21:25:22 UTC] Patriots   0.5700/0.5800 last 0.5800 | Broncos    0.4200/0.4300 last 0.4300
[21:25:30 UTC] Patriots   0.5800/0.5900 last 0.5800 | Broncos    0.4200/0.4300 last 0.4300
[21:25:32 UTC] Patriots   0.5800/0.5900 last 0.5800 | Broncos    0.4100/0.4200 last 0.4200
[21:25:37 UTC] Pat

## More Polymarket API features

This starter kit covers the **core building blocks** for the contest:
- discovering sports series/events via the **Gamma API**
- pulling current prices and order books via **CLOB** (`/price`, `/book`)
- optional real-time updates via the public **WebSocket** stream

If you need **more features beyond the base examples here** (additional endpoints and other WebSocket channels, etc.), please refer to the **official Polymarket API documentation**:

https://docs.polymarket.com/quickstart/overview#apis-at-a-glance
